## Silver Layer Transformations

## Read Silver Parquet File From DBFS

In [0]:
from pyspark.sql.functions import col, trim, lower, current_timestamp

CATALOG = "global_sales_analytics"
SCHEMA = "sales_analytics"

sales_df = spark.read.parquet("dbfs:/FileStore/silver/sales_silver.parquet")

## TABLE DROP IF EXISTS

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.silver_sales")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.dim_product")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.dim_customer")

DataFrame[]

In [0]:
# Standardize Column Names
sales_df = sales_df.toDF(*[c.lower().strip() for c in sales_df.columns])
display(sales_df)

order_id,order_date,product_id,customer_id,region,quantity,unit_price,cost_price,revenue,profit,profit_margin
1,2024-03-11T00:00:00.000,P104,C1061,South,7,200,150,1400,350,0.25
2,2024-12-01T00:00:00.000,P105,C1863,South,8,500,350,4000,1200,0.3
3,2024-09-06T00:00:00.000,P106,C0954,East,5,150,100,750,250,0.3333333333333333
4,2023-08-03T00:00:00.000,P108,C0800,East,3,1000,700,3000,900,0.3
5,2024-01-08T00:00:00.000,P101,C0423,North,8,120,80,960,320,0.3333333333333333
6,2024-10-25T00:00:00.000,P102,C1358,West,4,300,200,1200,400,0.3333333333333333
7,2024-10-23T00:00:00.000,P107,C1608,North,3,80,50,240,90,0.375
8,2023-12-10T00:00:00.000,P105,C0188,North,5,500,350,2500,750,0.3
9,2024-01-21T00:00:00.000,P105,C1242,North,6,500,350,3000,900,0.3
10,2023-06-10T00:00:00.000,P104,C1113,West,5,200,150,1000,250,0.25


## Data Cleaning

In [0]:
sales_df = sales_df.dropDuplicates()
sales_df = sales_df.dropna(subset=["order_id", "product_id", "customer_id"])

sales_df = sales_df.withColumn("region", lower(trim(col("region"))))
sales_df = sales_df.withColumn("order_date", col("order_date").cast("timestamp"))

sales_df = (
    sales_df
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("cost_price", col("cost_price").cast("double"))
    .withColumn("revenue", col("revenue").cast("double"))
    .withColumn("profit", col("profit").cast("double"))
)

sales_df = sales_df.withColumn("created_timestamp", current_timestamp())

sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_sales")



## Silver Layer Data

In [0]:
%sql
SELECT * FROM global_sales_analytics.sales_analytics.silver_sales LIMIT 10;

order_id,order_date,product_id,customer_id,region,quantity,unit_price,cost_price,revenue,profit,profit_margin,created_timestamp
44,2023-10-16T00:00:00.000Z,P101,C1382,south,3,120.0,80.0,360.0,120.0,0.3333333333333333,2026-03-02T13:38:53.795Z
59,2024-04-29T00:00:00.000Z,P106,C0353,north,4,150.0,100.0,600.0,200.0,0.3333333333333333,2026-03-02T13:38:53.795Z
61,2024-06-08T00:00:00.000Z,P102,C1597,north,4,300.0,200.0,1200.0,400.0,0.3333333333333333,2026-03-02T13:38:53.795Z
71,2024-09-19T00:00:00.000Z,P102,C0416,south,9,300.0,200.0,2700.0,900.0,0.3333333333333333,2026-03-02T13:38:53.795Z
74,2024-02-15T00:00:00.000Z,P102,C1181,east,8,300.0,200.0,2400.0,800.0,0.3333333333333333,2026-03-02T13:38:53.795Z
105,2024-10-01T00:00:00.000Z,P105,C1705,north,1,500.0,350.0,500.0,150.0,0.3,2026-03-02T13:38:53.795Z
135,2024-04-14T00:00:00.000Z,P106,C1992,south,3,150.0,100.0,450.0,150.0,0.3333333333333333,2026-03-02T13:38:53.795Z
166,2024-02-12T00:00:00.000Z,P103,C0805,south,3,50.0,30.0,150.0,60.0,0.4,2026-03-02T13:38:53.795Z
175,2023-11-01T00:00:00.000Z,P108,C0711,west,6,1000.0,700.0,6000.0,1800.0,0.3,2026-03-02T13:38:53.795Z
200,2023-04-04T00:00:00.000Z,P106,C1923,west,1,150.0,100.0,150.0,50.0,0.3333333333333333,2026-03-02T13:38:53.795Z


## Create Product Dimension Table 

In [0]:
dim_product = (
    sales_df
    .select("product_id")
    .dropDuplicates(["product_id"])
)

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_product")

## Dim Proucts Table Data

In [0]:
%sql
SELECT * FROM global_sales_analytics.sales_analytics.dim_product;

product_id
P101
P106
P103
P104
P105
P102
P108
P107


## Customer Dimension Table Creation

In [0]:
dim_customer = (
    sales_df
    .select("customer_id")
    .dropDuplicates(["customer_id"])
)

dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_customer")

## Dim Customer Table Data

In [0]:
%sql
SELECT * FROM global_sales_analytics.sales_analytics.dim_customer LIMIT 20;

customer_id
C1639
C1257
C1230
C0377
C1705
C1372
C0857
C0287
C0209
C1810


In [0]:
print("Silver Layer Created Successfully")

Silver Layer Created Successfully
